# Calibrate objective pair

Determine the ofset in XY and Z offset between two objectives
So switch between the objects does to change the sample is observed

Run once per objective pair.


## Objectives are relative — no fixed reference

This calibrates a **pair**: objective 1 (from) vs objective 2 (to). There is no
privileged, global reference objective. Translations are stored relative, so on a
fresh calibration the **first objective used** becomes the [0, 0, 0] origin, and
every other objective is placed relative to it.

**Best practice:** always validate each objective against one common objective
(e.g. measure 1->2 and 1->3). You never have to measure 2->3 directly — the code
derives any pair by composition (translate between objectives subtracts their two
translations). Measuring against a shared objective keeps every pair consistent.


## Before you start

1. Set up the rig first: run `set_stage_limits.ipynb` (physical envelope),
   then `set_orientation.ipynb` (image->stage rotation, applied at save time),
   then this notebook. The objective-pair run needs those in place, but does
   not read the orientation itself -- calibration frames are already
   stage-aligned when they are saved.
2. Steps 3a and 3b take XY images: they must use the **same zoom**.
   The two z-stacks (Steps 2a, 2b) can use any zoom.
3. After each Brenner measurement, the workflow parks z-wide at the
   fitted focus. You do not refocus manually between steps.
4. Mount the reference objective in LAS X before Step 2a.


## Step 1: Configure

Set the session ID and the two objectives. Edit `SESSIONS_ROOT` only if a
different data location is needed. `calibration_path` records the source
calibration for the report's provenance.

This cell opens LAS X and creates the session folder. Nothing is acquired.


In [ ]:
import _bootstrap
from navigator_expert.calibration.core import objective_pair as wf
from navigator_expert.config.machine import MACHINE

SESSIONS_ROOT = MACHINE.snapshot_root() / "calibration"
CALIBRATION_NAME = "10x_20x_water"  # edit per lens setup

session = wf.start_session(
    session_id="2026-05-22_scope_calibration",
    job_name="Overview",
    from_objective="10x",
    to_objective="20x",
    sessions_root=SESSIONS_ROOT,
    calibration_name=CALIBRATION_NAME,
)
print(session)

## Step 2a: Reference focus

Reference objective active. Focus on the sample and set up a z-stack
in LAS X. Run the cell.


In [ ]:
session = wf.measure_parfocality_reference(session)

## Step 2b: Target focus

Switch to the target objective. **Do not adjust z-wide manually** -- the
workflow reads the post-switch z-wide first. Set up a z-stack and run
the cell.


In [ ]:
session = wf.measure_parfocality_target(session)

## Step 3a: Reference XY image

Switch back to the reference objective. Pick a zoom for the XY image --
**Step 3b must use the same zoom.** Run the cell.

The workflow parks z-wide at the reference focus and acquires one image
at home XY.


In [ ]:
session = wf.measure_parcentricity_reference(session)

## Step 3b: Target XY image and save

Switch to the target objective. Same zoom as Step 3a. **Do not move the
XY stage manually** -- the workflow reads the post-switch XY first.
Run the cell.

The workflow parks z-wide at the target focus, acquires at the
post-switch XY, registers against the reference image, and writes the
calibration if the registration vote is trusted.


In [ ]:
summary = wf.measure_parcentricity_target_and_save(session)

## Step 4: Adopt

Run this cell only if the summary above shows `OK` and the overlay looks
correct. It publishes the calibration as a dated **snapshot** under
`C:\ProgramData\zmart-microscopy\...` — carrying the current stage limits
forward and archiving this executed notebook. The previous snapshot is kept as
history; the driver reads the newest.

In [ ]:
from navigator_expert.calibration.core import adopt

adopt.adopt_calibration(
    session,
    staging_name=session.objective_config_name,
    calibration_name=session.calibration_name,
    notebook_paths=[_bootstrap.NOTEBOOKS_DIR / "calibrate_objective_pair.ipynb"],
)